In [ ]:
# ffmpeg -i /home/aistudio/models/realityscan/Video/shitang.mp4 -r 15 -vf "scale='min(1024,iw)':'min(1024,ih)':force_original_aspect_ratio=decrease" -qscale:v 1 /home/aistudio/models/realityscan/Video/shitangframes/%04d.jpg

In [1]:
import pycolmap
from pathlib import Path

# 1. Setup Absolute Paths
# Based on your root /home/aistudio

image_path =  Path("/home/aistudio/models/realityscan/Video/shitangframes")
output_path =  Path("/home/aistudio/models/ColmapOutputs/Shitang")
output_path.mkdir(parents=True, exist_ok=True)

database_folder = output_path / "database"
database_folder.mkdir(exist_ok=True)
database_path = database_folder / "database.db"

# Create output directories

sparse_path = output_path / "sparse"
sparse_path.mkdir(exist_ok=True)

reader_options = pycolmap.ImageReaderOptions()


sift_options = pycolmap.SiftExtractionOptions()

sift_options.max_num_features = 8192
# 2. Feature Extraction
# This identifies key points in your 400 images
print(f"Extracting features from: {image_path}")
pycolmap.extract_features(
    database_path=str(database_path),
    image_path=str(image_path),

)

Extracting features from: /home/aistudio/models/realityscan/Video/shitangframes


I20260331 18:57:11.041049 140384608335616 feature_extraction.cc:502] === Feature extraction ===
I20260331 18:57:11.131310 140383790872320 sift.cc:754] Creating SIFT GPU feature extractor
I20260331 18:57:11.132841 140383782479616 feature_extraction.cc:280] Processed file [1/437]
I20260331 18:57:11.132879 140383782479616 feature_extraction.cc:283]   Name:            .ipynb_checkpoints/0003-checkpoint.jpg
W20260331 18:57:11.132883 140383782479616 feature_extraction.cc:287] .ipynb_checkpoints/0003-checkpoint.jpg IMAGE_EXISTS: Features for image were already extracted.
I20260331 18:57:11.133044 140383782479616 feature_extraction.cc:280] Processed file [2/437]
I20260331 18:57:11.133062 140383782479616 feature_extraction.cc:283]   Name:            .ipynb_checkpoints/0032-checkpoint.jpg
W20260331 18:57:11.133064 140383782479616 feature_extraction.cc:287] .ipynb_checkpoints/0032-checkpoint.jpg IMAGE_EXISTS: Features for image were already extracted.
I20260331 18:57:11.133074 140383782479616 fea

In [4]:

# 3. Feature Matching
# Exhaustive matching compares every image to every other image.
# For 400 images, this is thorough and reliable for 3D reconstruction.
print("Matching features...")
pycolmap.match_exhaustive(database_path=database_path)


Matching features...


I20260331 19:15:57.766206 140383799265024 feature_matching.cc:195] === Feature matching & geometric verification ===
I20260331 19:15:57.768879 140383816050432 feature_matching_utils.cc:80] Bind FeatureMatcherWorker to GPU device 0
I20260331 19:15:57.770268 140383816050432 sift.cc:1547] Creating SIFT GPU feature matcher
I20260331 19:15:57.777589 140383799265024 pairing.cc:180] Generating exhaustive image pairs...
I20260331 19:15:57.777612 140383799265024 pairing.cc:213] Processing block [1/9, 1/9]
I20260331 19:15:57.786007 140383799265024 feature_matching.cc:217] in 0.008s
I20260331 19:15:57.786038 140383799265024 pairing.cc:213] Processing block [1/9, 2/9]
I20260331 19:15:57.793914 140383799265024 feature_matching.cc:217] in 0.008s
I20260331 19:15:57.793940 140383799265024 pairing.cc:213] Processing block [1/9, 3/9]
I20260331 19:15:57.802011 140383799265024 feature_matching.cc:217] in 0.008s
I20260331 19:15:57.802035 140383799265024 pairing.cc:213] Processing block [1/9, 4/9]
I20260331

In [5]:

# 4. Reconstruction (Structure from Motion)
# This calculates the 3D points and camera positions.
print("Starting incremental reconstruction...")
reconstructions = pycolmap.incremental_mapping(
    database_path=str(database_path),
    image_path=str(image_path),
    output_path=sparse_path
)

# 5. Finalize for Gaussian Splatting
# Gaussian Splatting usually expects the '0' folder (points3D.bin, cameras.bin, images.bin)
if reconstructions:
    print(f"Success! Found {len(reconstructions)} reconstruction clusters.")
    # Export the primary model to the '0' subfolder
    final_output = sparse_path / "0"
    final_output.mkdir(exist_ok=True)
    reconstructions[0].write(str(final_output))
    print(f"Model saved to: {final_output}")
else:
    print("Reconstruction failed. Check if your images have enough overlap.")

Starting incremental reconstruction...


I20260331 19:16:10.437371 140389279659840 incremental_pipeline.cc:265] Loading database
I20260331 19:16:10.437478 140389279659840 database_cache.cc:72] Loading rigs...
I20260331 19:16:10.437777 140389279659840 database_cache.cc:82]  437 in 0.000s
I20260331 19:16:10.437792 140389279659840 database_cache.cc:90] Loading cameras...
I20260331 19:16:10.438379 140389279659840 database_cache.cc:108]  437 in 0.001s
I20260331 19:16:10.438389 140389279659840 database_cache.cc:116] Loading frames...
I20260331 19:16:10.438936 140389279659840 database_cache.cc:126]  437 in 0.001s
I20260331 19:16:10.438962 140389279659840 database_cache.cc:134] Loading matches...
I20260331 19:16:11.468955 140389279659840 database_cache.cc:139]  95266 in 1.030s
I20260331 19:16:11.469018 140389279659840 database_cache.cc:147] Loading images...
I20260331 19:16:11.849028 140389279659840 database_cache.cc:241]  437 in 0.380s (connected 437, loaded 437)
I20260331 19:16:11.849150 140389279659840 database_cache.cc:255] Loadi

In [ ]:
import pycolmap
import numpy as np

# Path to your COLMAP sparse folder
sparse_folder = "/home/aistudio/models/ColmapOutputs/Shitang/sparse/0"

# Load reconstruction
recon = pycolmap.Reconstruction(sparse_folder)
# recon.cameras is the mapping of CameraID -> Camera object
print(f"Number of cameras found: {len(recon.cameras)}")

for camera_id, camera in recon.cameras.items():
    print(f"\n--- Camera ID: {camera_id} ---")
    print(f"Model: {camera.model_name}")
    print(f"Dimensions: {camera.width} x {camera.height}")
    
    # These are the raw values from cameras.bin
    params = camera.params 
    print(f"Raw Parameters: {params}")

# points3D is a dict: {point3D_id: Point3D_object}
print(f"Total points in sparse cloud: {len(recon.points3D)}")

# Let's look at the first few points
for i, (point_id, point) in enumerate(recon.points3D.items()):
    if i >= 3: break
    
    xyz = point.xyz      # [x, y, z] coordinates
    rgb = point.color    # [r, g, b] (0-255)
    error = point.error  # Reprojection error (lower is better)
    
    print(f"Point {point_id}: Pos={xyz}, Color={rgb}, Error={error:.4f}")